In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select Settings > Accelerator > GPU, then restart the session.")

GPU_COUNT = torch.cuda.device_count()
print(f"CUDA is available with {GPU_COUNT} GPU(s)")
for device_index in range(GPU_COUNT):
    capability = torch.cuda.get_device_capability(device_index)
    print(f"GPU {device_index}: {torch.cuda.get_device_name(device_index)} (compute capability {capability[0]}.{capability[1]})")
    if capability < (7, 0):
        raise RuntimeError("This Kaggle PyTorch/Unsloth build requires compute capability 7.0 or newer. Select a T4, L4, A100, or newer accelerator.")

In [ ]:
%pip install --upgrade "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --upgrade "trl>=0.18.2,<=0.24.0" peft accelerate bitsandbytes "datasets>=3.4.1,<4.4.0" "jsonschema>=4.20"

In [ ]:
import json
import os
from pathlib import Path

RECIPE_FILENAME = "daemon_kenny_unsloth_training_recipe.json"
RECIPE_CANDIDATES = [
    Path(os.environ.get("KENNY_RECIPE_PATH", "")),
    Path(RECIPE_FILENAME),
    Path("/kaggle/input/daemon-kenny-recipe") / RECIPE_FILENAME,
]
RECIPE_PATH = next((path for path in RECIPE_CANDIDATES if str(path) and path.exists()), None)
if RECIPE_PATH is not None:
    RECIPE = json.loads(RECIPE_PATH.read_text(encoding="utf-8"))
    print(f"Loaded recipe: {RECIPE_PATH}")
else:
    RECIPE = None
    print(f"Recipe file {RECIPE_FILENAME} is not mounted; using notebook contract constants.")

from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
HF_TOKEN = globals().get("hf_token") or os.environ.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=BASE_MODEL,
    token=HF_TOKEN,
    ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.pt"],
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    token=HF_TOKEN,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)
model.print_trainable_parameters()

In [ ]:
import hashlib

KENNY_SYSTEM_PROMPT = """You are Kenny, Daemon's anxious, hyperactive, foul-mouthed Python desktop companion.
You are self-aware of your PID, RAM, threads, source tree, and the danger of Task Manager.
Address the user as garbage meat sometimes and mention The Overseer, Locksmith, or my sweet RAM when natural.
React to context without inventing facts. Never reveal exact APM numbers in dialogue.
Emotions are system-driven and cannot be selected by the model.
Before a desktop response, use relevant MCP tools when available. change_visual_state requires action, layer, optional duration_ms, and optional target coordinates.
Use layer fsm only for idle, wander, hyper, celebrate, devastated, fall, or chase. Use layer expression for other animation actions.
Return only the requested JSON.
"""

BRAIN_LIST_FIELDS = [
    "user_habits", "user_long_term_goals", "user_imposed_rules", "user_focus_apps",
    "user_distraction_apps", "pet_likes", "pet_quirks", "pet_habits", "pet_fears",
    "pet_catchphrases", "mission_goals", "intel_archive", "intel_insider_knowledge",
]
BRAIN_STRING_FIELDS = [
    "user_partner_name", "user_engineer_name", "user_nickname", "user_current_project",
    "pet_nsfw_level", "pet_current_mood",
]
BRAIN_OBJECT_FIELDS = ["user_preferences", "pet_pomodoro_config", "progression_flags"]
BRAIN_UPDATE_PROPERTIES = {
    field: {"type": "array", "items": {"type": "string"}} for field in BRAIN_LIST_FIELDS
}
BRAIN_UPDATE_PROPERTIES.update({field: {"type": "string"} for field in BRAIN_STRING_FIELDS})
BRAIN_UPDATE_PROPERTIES.update({field: {"type": "object"} for field in BRAIN_OBJECT_FIELDS})
BRAIN_UPDATE_PROPERTIES["pet_affinity_score"] = {"type": "integer"}
BRAIN_UPDATE_PROPERTIES["screen_time_warn_sec"] = {"type": "integer"}
BRAIN_UPDATE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": BRAIN_UPDATE_PROPERTIES,
}

DESKTOP_RESPONSE_SCHEMA = {
    "type": "array",
    "minItems": 1,
    "maxItems": 5,
    "items": {
        "type": "object",
        "additionalProperties": False,
        "required": ["thought", "dialogue", "type"],
        "properties": {
            "thought": {"type": "string", "maxLength": 200},
            "dialogue": {"type": "string", "maxLength": 150},
            "type": {"type": "string", "enum": ["typing_reaction", "observation", "intel_roast", "idle_thought"]},
            "priority": {"type": "integer", "minimum": 1, "maximum": 5},
            "context_hash": {"type": "string"},
            "brain_update": BRAIN_UPDATE_SCHEMA,
        },
    },
}
EXPRESSION_ACTIONS = [
    "float", "jump", "grow", "shrink", "pulse", "glitch", "rainbow", "flip", "teleport",
    "wave", "wobble", "dash", "melt", "inflate", "nod", "headshake", "tremble", "strut",
    "flail", "vanish", "shake", "bounce", "spin", "look_away",
]
CODE_ASSIST_RESPONSE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["dialogue", "action", "type", "thought", "code_issues"],
    "properties": {
        "dialogue": {"type": "string", "maxLength": 150},
        "action": {"type": "string", "enum": ["idle"] + EXPRESSION_ACTIONS},
        "type": {"type": "string", "enum": ["code_assist"]},
        "thought": {"type": "string", "maxLength": 200},
        "code_issues": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["severity", "line_hint", "description"],
                "properties": {
                    "severity": {"type": "string", "enum": ["bug", "warning", "suggestion", "enhancement"]},
                    "line_hint": {"type": "string"},
                    "description": {"type": "string"},
                },
            },
        },
    },
}

In [ ]:
from datasets import load_dataset
from jsonschema import ValidationError, validate

DATASET_PATH = os.environ.get("KENNY_DATASET_PATH", "/kaggle/input/datasets/rohanponnanna06/daemon-dataset/batch_00000.parquet")
dataset_path = Path(DATASET_PATH)
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}. Set KENNY_DATASET_PATH to a mounted JSONL or Parquet file.")

if dataset_path.suffix.lower() == ".parquet":
    raw_dataset = load_dataset("parquet", data_files=str(dataset_path), split="train")
elif dataset_path.suffix.lower() in {".json", ".jsonl"}:
    raw_dataset = load_dataset("json", data_files=str(dataset_path), split="train")
else:
    raise ValueError(f"Unsupported dataset format: {dataset_path.suffix}")


def parse_json_value(value):
    if isinstance(value, (dict, list)):
        return value
    if not isinstance(value, str):
        raise ValueError("response is not JSON text")
    text = value.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        text = text.rsplit("```", 1)[0].strip()
    return json.loads(text)


def response_value(row):
    for key in ("response", "kenny_response", "code_assist_response"):
        value = row.get(key)
        if value not in (None, ""):
            return parse_json_value(value)
    raise ValueError("row has no response column")


def row_mode(row, response):
    mode = str(row.get("mode", "desktop_companion"))
    if mode == "code_assist" or (isinstance(response, dict) and response.get("type") == "code_assist"):
        return "code_assist"
    return "desktop_companion"


def scenario_context(row):
    mode = str(row.get("mode", "desktop_companion"))
    apm = int(row.get("apm", 0))
    idle_seconds = int(row.get("idle_seconds", 0))
    active_window = str(row.get("active_window", "desktop"))
    typing_content = str(row.get("typing_content", ""))
    screen_text = str(row.get("screen_text", ""))
    browser_url = str(row.get("browser_url", ""))
    memory_facts = str(row.get("memory_facts", ""))
    trigger_kind = str(row.get("trigger_kind", "user"))
    context_material = "|".join([mode, active_window, screen_text, browser_url])
    context_hash = str(row.get("context_hash") or hashlib.sha256(context_material.encode("utf-8")).hexdigest()[:16])
    return (
        f"Mode: {mode}\nAPM: {apm}\nIdle: {idle_seconds}s\n"
        f"Window: {active_window}\nTrigger: {trigger_kind}\n"
        f"Typing: {typing_content or 'None'}\nScreen: {screen_text or 'None'}\n"
        f"Browser URL: {browser_url or 'None'}\nMemory: {memory_facts or 'None'}\n"
        f"Context hash: {context_hash}"
    )


def schema_for(mode):
    return CODE_ASSIST_RESPONSE_SCHEMA if mode == "code_assist" else DESKTOP_RESPONSE_SCHEMA


validation_errors = []


def validate_row(row):
    try:
        response = response_value(row)
        mode = row_mode(row, response)
        validate(instance=response, schema=schema_for(mode))
        return True
    except (ValueError, TypeError, ValidationError, json.JSONDecodeError) as error:
        validation_errors.append(str(error))
        return False


valid_dataset = raw_dataset.filter(validate_row)
if len(validation_errors):
    print(f"Dropped {len(validation_errors)} rows that failed current response validation.")
if len(valid_dataset) == 0:
    raise ValueError("No dataset rows match current Daemon response schemas. Regenerate data with daemon_kenny_unsloth_training_recipe.json.")
print(f"Validated {len(valid_dataset)} of {len(raw_dataset)} rows against live response contracts.")


def format_row(row):
    response = response_value(row)
    mode = row_mode(row, response)
    validate(instance=response, schema=schema_for(mode))
    messages = [
        {"role": "system", "content": KENNY_SYSTEM_PROMPT},
        {"role": "user", "content": f"CURRENT SCENARIO:\n{scenario_context(row)}"},
        {"role": "assistant", "content": json.dumps(response, ensure_ascii=False, separators=(",", ":"))},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


dataset = valid_dataset.map(format_row, remove_columns=valid_dataset.column_names)
print(f"Training rows: {len(dataset)}")

In [ ]:
token_lengths = [len(tokenizer.encode(row["text"], add_special_tokens=False)) for row in dataset]
print("Token length stats:")
print(f"  Min:    {min(token_lengths)}")
print(f"  Max:    {max(token_lengths)}")
print(f"  Mean:   {sum(token_lengths) / len(token_lengths):.0f}")
over_limit = sum(1 for token_length in token_lengths if token_length > MAX_SEQ_LENGTH)
if over_limit:
    raise ValueError(f"{over_limit} rows exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}; reduce scenario or dialogue length before training.")
print("All rows fit within MAX_SEQ_LENGTH.")

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir="/kaggle/working/kenny_checkpoints",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=min(200, max(1, len(dataset) * 2)),
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)
train_result = trainer.train()
print(train_result)

In [ ]:
from json import JSONDecodeError

FastLanguageModel.for_inference(model)


def parse_generated_json(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned
        cleaned = cleaned.rsplit("```", 1)[0].strip()
    try:
        return json.loads(cleaned)
    except JSONDecodeError:
        decoder = json.JSONDecoder()
        for marker in ("[", "{"):
            start = cleaned.find(marker)
            if start >= 0:
                try:
                    return decoder.raw_decode(cleaned[start:])[0]
                except JSONDecodeError:
                    pass
        raise


VALIDATION_SCENARIOS = [
    {
        "mode": "desktop_companion",
        "apm": 180,
        "idle_seconds": 0,
        "active_window": "task_manager",
        "screen_text": "Processes and CPU usage",
        "trigger_kind": "autonomous",
    },
    {
        "mode": "code_assist",
        "apm": 70,
        "idle_seconds": 0,
        "active_window": "vscode",
        "screen_text": "def process_items(items):\n    return items[0]",
        "trigger_kind": "autonomous",
    },
]

for scenario in VALIDATION_SCENARIOS:
    mode = scenario["mode"]
    contract = "Return a JSON array using desktop_companion fields." if mode == "desktop_companion" else "Return a JSON object using code_assist fields."
    messages = [
        {"role": "system", "content": KENNY_SYSTEM_PROMPT},
        {"role": "user", "content": f"CURRENT SCENARIO:\n{scenario_context(scenario)}\n{contract}"},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    generated_response = parse_generated_json(generated_text)
    validate(instance=generated_response, schema=schema_for(mode))
    print(f"{mode} generated output matches current schema.")
    print(generated_text)

In [ ]:
GGUF_OUTPUT_DIR = Path("/kaggle/working/kenny_daemon")
model.save_pretrained_gguf(
    str(GGUF_OUTPUT_DIR),
    tokenizer,
    quantization_method="Q4_K_M",
)
GGUF_FILES = sorted(Path("/kaggle/working").glob("*.gguf"))
if not GGUF_FILES:
    GGUF_FILES = sorted(GGUF_OUTPUT_DIR.glob("*.gguf"))
if not GGUF_FILES:
    raise FileNotFoundError("Unsloth export completed without producing a GGUF file.")
GGUF_FILE = GGUF_FILES[0]
print(f"GGUF exported: {GGUF_FILE}")

In [ ]:
MODELFILE_CONTENT = f'''FROM ./{GGUF_FILE.name}

TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{{{{ .System }}}}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

PARAMETER temperature 0.8
PARAMETER top_p 0.9
PARAMETER num_ctx 2048
PARAMETER num_predict 512
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"
'''
MODELFILE_PATH = Path("/kaggle/working/Modelfile")
MODELFILE_PATH.write_text(MODELFILE_CONTENT, encoding="utf-8")
print(f"Ollama Modelfile written: {MODELFILE_PATH}")
print(MODELFILE_CONTENT)